In [1]:
from kfp import dsl
from kfp.dsl import (
    Input, Output, Dataset, Model, component, pipeline
)
from google.cloud import aiplatform
from utils.components import vertex_hyperparameter_tuning_component,custom_job_training_component,upload_model_component,vertex_batch_predict_bigquery_component
from typing import Any, Dict, List, Optional, Text, Tuple, Union, Sequence, Callable
import yaml
from kfp.v2 import compiler

c:\Program Files\Python\313\Lib\site-packages\kfp\dsl\component_decorator.py:126: FutureWarning: The default base_image used by the @dsl.component decorator will switch from 'python:3.9' to 'python:3.10' on Oct 1, 2025. To ensure your existing components work with versions of the KFP SDK released after that date, you should provide an explicit base_image argument and ensure your component works as intended on Python 3.10.
  return component_factory.create_component_from_func(


In [1]:
import re
import pandas as pd
def is_date_column(series, sample_size=1000):
    """
    Enhanced date detection for pandas Series.
    
    Args:
        series: pandas Series to check
        sample_size: number of values to sample for analysis
    
    Returns:
        bool: True if the series appears to contain dates
    """
    # Sample the data for faster processing
    sample_data = series.dropna().head(sample_size)
    
    if len(sample_data) == 0:
        return False
    
    # Method 1: Check pandas datetime dtype
    if pd.api.types.is_datetime64_any_dtype(series):
        return True
    
    # Method 1.5: Check for BigQuery/Google Cloud date types
    dtype_str = str(series.dtype)
    if 'DateDtype' in dtype_str or 'db_dtypes' in dtype_str:
        return True
    
    # Method 2: Check if column name suggests it's a date
    date_keywords = ['date', 'time', 'dt', 'timestamp', 'created', 'updated', 'dob', 'birth']
    col_name_lower = series.name.lower()
    if any(keyword in col_name_lower for keyword in date_keywords):
        return True
    
    # Method 3: Try to parse as datetime
    try:
        # Check if values look like dates (common formats)
        sample_str = sample_data.astype(str).iloc[0]
        
        # Common date patterns
        date_patterns = [
            r'\d{4}-\d{2}-\d{2}',  # YYYY-MM-DD
            r'\d{2}/\d{2}/\d{4}',  # MM/DD/YYYY
            r'\d{2}-\d{2}-\d{4}',  # MM-DD-YYYY
            r'\d{4}/\d{2}/\d{2}',  # YYYY/MM/DD
            r'\d{8}',              # YYYYMMDD
        ]
        
        import re
        if any(re.match(pattern, sample_str) for pattern in date_patterns):
            # Try to convert a sample to datetime
            pd.to_datetime(sample_data.head(5), errors='raise')
            return True
            
    except (ValueError, TypeError, pd.errors.ParserError):
        pass
    
    # Method 4: Check if numeric values could be timestamps
    if pd.api.types.is_numeric_dtype(series):
        sample_values = sample_data.head(10)
        # Check if values look like Unix timestamps or Excel dates
        if (sample_values > 1000000000).all() and (sample_values < 2000000000).all():
            return True
        # Check if values look like Excel date serial numbers
        if (sample_values > 40000).all() and (sample_values < 50000).all():
            return True
    
    return False

def process_sql_file(sql_content, constants):
    # Find all variables in the SQL file
    variables_in_sql = re.findall(r'\{([^}]+)\}', sql_content)
    
    # Create a clean substitution dictionary from constants
    substitution_dict = {}
    
    for key, value in constants.items():
        # Skip nested dictionaries and non-string values that aren't useful for SQL
        if isinstance(value, dict):
            continue  # Skip LABELS and other nested objects
        elif isinstance(value, bool):
            substitution_dict[key] = str(value).upper()  # Convert True/False to TRUE/FALSE
        else:
            substitution_dict[key] = str(value)  # Convert everything to string
    
    # Add additional mappings for common SQL variable patterns
    additional_mappings = {
        'COST_CENTER': constants.get('COSTCENTER', ''),  # Map COSTCENTER to COST_CENTER
    }
    
    # Merge additional mappings
    substitution_dict.update(additional_mappings)
    
    # Check which variables can be substituted
    missing_variables = []
    available_substitutions = {}
    
    for var_name in variables_in_sql:
        if var_name in substitution_dict:
            available_substitutions[var_name] = substitution_dict[var_name]
        else:
            missing_variables.append(var_name)
    
    # Warn about missing variables but don't fail
    if missing_variables:
        print(f"Warning: Variables not found in constants: {missing_variables}")
        print(f"Available substitutions: {list(substitution_dict.keys())}")
        print(f"Variables in SQL: {variables_in_sql}")
    
    # Substitute variables using **kwargs
    try:
        sql_query = sql_content.format(**available_substitutions)
    except KeyError as e:
        print(f"Available substitutions: {list(available_substitutions.keys())}")
        raise
    
    return sql_query

def auto_classify_features(df, unique_threshold_binary=2, unique_threshold_categorical=20, 
                          exclude_cols=['asdb_member_key', 'index_date', 'index_dt'], 
                          sample_size=100000, manual_overrides=None, verbose=False):
    """
    Automatically classify dataframe columns into:
    0 := categorical (discrete values, typically strings or low cardinality)
    1 := continuous (numeric with many unique values)
    2 := binary (exactly 2 unique values or boolean-like)
    3 := datetime (date/time columns)
    
    Args:
        df: pandas DataFrame to analyze
        unique_threshold_binary: max unique values to be considered binary (default: 2)
        unique_threshold_categorical: max unique values to be considered categorical (default: 20)
        exclude_cols: list of columns to exclude from classification
        sample_size: number of rows to sample for faster processing on large datasets
        manual_overrides: dict of manual overrides {column_name: type}
        verbose: print detailed information
    
    Returns:
        dict: mapping of feature_name -> type (0, 1, 2, or 3)
    """
    
    feature_types = {}
    
    # Sample if dataframe is large
    if len(df) > sample_size:
        df_sample = df.sample(n=sample_size, random_state=42)
    else:
        df_sample = df
    
    for col in df.columns:
        # Skip excluded columns
        if col in exclude_cols:
            feature_types[col] = 1  # or you can skip entirely
            continue
        
        # Apply manual overrides first
        if manual_overrides and col in manual_overrides:
            feature_types[col] = manual_overrides[col]
            if verbose:
                print(f"Manual override: {col} -> {manual_overrides[col]}")
            continue
        
        # Get non-null values for analysis
        non_null_values = df_sample[col].dropna()
        
        if len(non_null_values) == 0:
            feature_types[col] = 1  # Default for empty columns
            continue
        
        n_unique = non_null_values.nunique()
        dtype = df[col].dtype
        
        # Step 1: Check for DATETIME (3) - ENHANCED DETECTION
        if is_date_column(df[col]):
            feature_types[col] = 3
            if verbose:
                print(f"Date detected: {col} -> 3 (datetime)")
            continue
        
        # Step 2: Check for BINARY (2)
        if n_unique <= unique_threshold_binary:
            unique_vals = set(non_null_values.unique())
            if (unique_vals <= {0, 1} or 
                unique_vals <= {0.0, 1.0} or 
                unique_vals <= {True, False} or
                unique_vals <= {'Yes', 'No'} or
                unique_vals <= {'Y', 'N'} or
                unique_vals <= {1, 2}):
                feature_types[col] = 2
                if verbose:
                    print(f"Binary detected: {col} -> 2 (binary)")
                continue
        
        # Step 3: Check for CATEGORICAL (0)
        if dtype == 'object' or dtype.name == 'category':
            feature_types[col] = 0
            if verbose:
                print(f"Categorical detected: {col} -> 0 (categorical)")
        elif n_unique <= unique_threshold_categorical:
            # Low cardinality numeric could be categorical
            feature_types[col] = 0
            if verbose:
                print(f"Low cardinality categorical: {col} -> 0 (categorical, {n_unique} unique values)")
        else:
            # Step 4: CONTINUOUS (1)
            feature_types[col] = 1
            if verbose:
                print(f"Continuous detected: {col} -> 1 (continuous)")
    
    return feature_types


In [2]:
import pandas_gbq
import pandas as pd
from concurrent.futures import ThreadPoolExecutor
import numpy as np

def read_bq_chunk(query, project_id, chunk_params, **kwargs):
    """Read a specific chunk of data from BigQuery"""
    # Add LIMIT and OFFSET to query for chunking
    if 'LIMIT' not in query.upper():
        modified_query = f"{query} LIMIT {chunk_params['limit']} OFFSET {chunk_params['offset']}"
    else:
        modified_query = query  # Don't modify if LIMIT already exists
    
    print(f"Reading chunk: OFFSET {chunk_params['offset']}, LIMIT {chunk_params['limit']}")
    
    return pandas_gbq.read_gbq(
        modified_query,
        project_id=project_id,
        use_bqstorage_api=True,
        progress_bar_type='tqdm',  # Disable individual progress bars
        **kwargs
    )

def read_bigquery_parallel(base_query, user_constants, project_id="anbc-dev-hcm-cm-de", 
                         max_workers=4, chunk_size=500000, **kwargs):
    """
    Read BigQuery data in parallel chunks
    
    Args:
        base_query: SQL query to execute
        user_constants: Constants dictionary
        project_id: GCP project ID
        max_workers: Number of parallel workers
        chunk_size: Number of rows per chunk
        **kwargs: Additional arguments for pandas_gbq.read_gbq()
    """
    # Process SQL with constants
    processed_query = process_sql_file(base_query, user_constants)
    
    # First, get total count to determine chunks
    count_query = f"SELECT COUNT(*) as cnt FROM ({processed_query})"
    count_df = pandas_gbq.read_gbq(
        count_query,
        project_id=project_id,
        use_bqstorage_api=True,
        progress_bar_type='tqdm',
    )
    total_rows = count_df['cnt'].iloc[0]
    print(f"Total rows to read: {total_rows}")
    
    # Calculate number of chunks
    num_chunks = min(max_workers, int(np.ceil(total_rows / chunk_size)))
    print(f"Reading in {num_chunks} parallel chunks...")
    
    # Create chunk parameters
    chunks = []
    for i in range(num_chunks):
        chunks.append({
            'offset': i * chunk_size,
            'limit': chunk_size
        })
    
    # Read chunks in parallel
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = [
            executor.submit(read_bq_chunk, processed_query, project_id, chunk, **kwargs)
            for chunk in chunks
        ]
        
        # Collect results
        results = []
        for future in futures:
            result = future.result()
            if not result.empty:
                results.append(result)
                print(f"✅ Read chunk with {len(result)} rows")
    
    # Combine all results
    if results:
        final_df = pd.concat(results, ignore_index=True)
        print(f"✅ Total rows read: {len(final_df)}")
        return final_df
    else:
        print("⚠️ No data returned")
        return pd.DataFrame()

# Usage in your notebook:
pd.set_option('mode.chained_assignment', None)
pd.set_option('compute.use_bottleneck', True)
pd.set_option('compute.use_numexpr', True)



In [3]:
user_constants = {
    # SQL Variables for BigQuery queries
    "GCP_PROJECT": "anbc-hcb-dev",
    "GCP_DB": "cm_medicaid_hcb_dev", 
    "PREFIX": "a974930_sahil_test",
    "DEFAULT_EXP": "INTERVAL 2 DAY",
    "ST": "{GCP_PROJECT}.{GCP_DB}.{PREFIX}_st",
    "SDOH_YEAR":'2023'
}

sql = """
WITH base_data AS (
  SELECT *
    , GREATEST(pre_term_labor_clm, pre_term_delivery_clm) as pre_term_max
  FROM `anbc-hcb-dev.cm_medicaid_hcb_dev.a974930_sahil_test_combined_features_train_data` 
),
filtered_data AS (
  SELECT * EXCEPT(
    asdb_member_key,
    index_date,
    preeclampsia,
    pre_term_labor_clm,
    pre_term_delivery_clm,
    diabetes,
    baby_dob,
    nicu_lvl,
    nicu_flag,
    post_mnths,
    index_dt
  )
  FROM base_data
)
SELECT * FROM filtered_data
"""
sql = """
SELECT * FROM `anbc-hcb-dev.cm_medicaid_hcb_dev.a974930_sahil_test_combined_features_train_data`
"""
# Get features
sql2 = """
WITH base_data AS (
    SELECT DISTINCT
        a.mom_key AS asdb_member_key
        , a.index_date
        , a.preeclampsia
        , a.diabetes
        , a.gest_age
        , b.* EXCEPT(asdb_member_key, index_date)
        , c.* EXCEPT(asdb_member_key, index_date, baby_dob)
        , d.* EXCEPT(asdb_member_key, asdb_plan_key, index_dt, oud, autoimmune)
        , e.* EXCEPT(individual_id)
        , GREATEST(pre_term_labor_clm, pre_term_delivery_clm) as pre_term_max
    FROM
    `{GCP_PROJECT}.{GCP_DB}.{PREFIX}_longitudinal` AS a
LEFT JOIN
    `{GCP_PROJECT}.{GCP_DB}.{PREFIX}_all_risk_w_nicu_flags` AS b
        ON a.mom_key = b.asdb_member_key
        AND a.index_date = b.index_date
LEFT JOIN
    `{GCP_PROJECT}.{GCP_DB}.{PREFIX}_all_labs_joined` AS c
        ON a.mom_key = c.asdb_member_key
        AND a.index_date = c.index_date
LEFT JOIN
    `{GCP_PROJECT}.{GCP_DB}.{PREFIX}_non_embedding_features` AS d
        ON a.mom_key = d.asdb_member_key
        AND a.index_date = d.index_dt
LEFT JOIN
    `anbc-hcb-prod.cm_medicaid_hcb_prod.medicaid_transformer_embed_scores_hist` AS e
        ON a.mom_key = e.individual_id
        AND DATE_TRUNC(a.index_date, MONTH) = DATE_TRUNC(e.index_dt, MONTH)
WHERE 1 = 1
    AND a.mom_key IN (SELECT mom_key FROM `{GCP_PROJECT}.{GCP_DB}.{PREFIX}_testing_ids`)    
    AND a.diabetes_at_index = 0  
     LIMIT 1000
),
filtered_data AS (
    SELECT * EXCEPT(
        asdb_member_key,
        index_date,
        preeclampsia,
        diabetes,
        baby_dob,
        nicu_lvl,
        pre_term_labor_clm,
        pre_term_delivery_clm,
        nicu_flag,
        post_mnths,
        index_dt
    )
    FROM base_data
)
SELECT * FROM filtered_data
"""
# Configure pandas to use multiple threads and optimize memory
# pd.set_option('mode.chained_assignment', None)
# pd.set_option('compute.use_bottleneck', True)
# pd.set_option('compute.use_numexpr', True)

# import pandas_gbq

# # The SQL query is the FIRST POSITIONAL argument, not query=
# df = pandas_gbq.read_gbq(
#     process_sql_file(sql, user_constants),  # First argument - the SQL query
#     project_id="anbc-dev-hcm-cm-de",
#     use_bqstorage_api=True,  # CRITICAL for speed
#     progress_bar_type='tqdm',
#     configuration={
#         'query': {
#             'useQueryCache': False,
#             'useLegacySql': False,
#         }
#     },
#     dialect='standard',
#     auth_local_webserver=False
# )
pd.set_option('mode.chained_assignment', None)
pd.set_option('compute.use_bottleneck', True)
pd.set_option('compute.use_numexpr', True)

# import pandas_gbq
# import tqdm
# df = pandas_gbq.read_gbq(
#     process_sql_file(sql, user_constants),  # First argument - the SQL query
#     project_id="anbc-dev-hcm-cm-de",
#     use_bqstorage_api=True,  # CRITICAL for speed
#     progress_bar_type='tqdm',
#     configuration={
#         'query': {
#             'useQueryCache': False,
#             'useLegacySql': False,
#         }
#     },
#     dialect='standard',
#     auth_local_webserver=False
# )
# print(df.shape)


In [4]:
df = read_bigquery_parallel(
    sql,  # Your SQL query
    user_constants,
    project_id="anbc-dev-hcm-cm-de",
    max_workers=6,  # Use 4 parallel workers
    chunk_size=100000,  # 500k rows per chunk
    configuration={
        'query': {
            'useQueryCache': False,
            'useLegacySql': False,
        }
    },
    dialect='standard',
    auth_local_webserver=False
)

print(df.shape)

Downloading: 100%|██████████|
Total rows to read: 1374123
Reading in 6 parallel chunks...
Reading chunk: OFFSET 0, LIMIT 100000
Reading chunk: OFFSET 100000, LIMIT 100000
Reading chunk: OFFSET 200000, LIMIT 100000
Reading chunk: OFFSET 300000, LIMIT 100000
Reading chunk: OFFSET 400000, LIMIT 100000
Reading chunk: OFFSET 500000, LIMIT 100000
Downloading:   0%|          |
Downloading:  10%|█         |

Downloading:  15%|█▍        |
Downloading:  15%|█▍        |

Downloading:  15%|█▌        |
Downloading:  16%|█▌        |

Downloading:  16%|█▋        |
Downloading:  17%|█▋        |
Downloading:  17%|█▋        |
Downloading:  17%|█▋        |
Downloading:  18%|█▊        |
Downloading:  18%|█▊        |
Downloading:  18%|█▊        |
Downloading:  18%|█▊        |
Downloading:  19%|█▊        |
Downloading:  19%|█▉        |
Downloading:  19%|█▉        |
Downloading:  20%|█▉        |
Downloading:  20%|█▉        |
Downloading:  20%|██        |
Downloading:  20%|██        |
Downloading:  21%|██    

In [ ]:
import pandas_gbq
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed
import numpy as np
from tqdm.auto import tqdm
import time

def clean_query(query):
    """Remove trailing semicolon from SQL query"""
    return query.rstrip(' ;')

def read_bq_chunk(query, project_id, chunk_params, chunk_num, total_chunks, **kwargs):
    """Read a specific chunk of data from BigQuery"""
    clean_query_str = clean_query(query)
    
    # Use CTE to compute row numbers, then filter in outer query
    chunk_query = f"""
    WITH numbered_data AS (
        SELECT *, ROW_NUMBER() OVER (ORDER BY (SELECT NULL)) as rn
        FROM ({clean_query_str})
    )
    SELECT * EXCEPT(rn)
    FROM numbered_data
    WHERE rn BETWEEN {chunk_params['start_row']} AND {chunk_params['end_row']}
    """
    
    try:
        result = pandas_gbq.read_gbq(
            chunk_query,
            project_id=project_id,
            use_bqstorage_api=True,
            progress_bar_type=None,
            **kwargs
        )
        return {
            'chunk_num': chunk_num,
            'result': result,
            'success': True,
            'rows': len(result) if not result.empty else 0,
            'message': f'Chunk {chunk_num}/{total_chunks} complete'
        }
    except Exception as e:
        return {
            'chunk_num': chunk_num,
            'result': pd.DataFrame(),
            'success': False,
            'rows': 0,
            'message': f'Chunk {chunk_num}/{total_chunks} failed: {str(e)}'
        }

def read_bigquery_parallel(base_query, user_constants, project_id="anbc-dev-hcm-cm-de", 
                         max_workers=4, chunk_size=100000, **kwargs):
    """
    Read BigQuery data in parallel chunks WITHOUT duplicates with progress tracking
    """
    # Process SQL with constants
    processed_query = process_sql_file(base_query, user_constants)
    processed_query = clean_query(processed_query)
    
    # First, get total count to determine chunks
    count_query = f"SELECT COUNT(*) as cnt FROM ({processed_query})"
    print("🔍 Counting total rows...")
    
    try:
        count_df = pandas_gbq.read_gbq(
            count_query,
            project_id=project_id,
            use_bqstorage_api=True,
            progress_bar_type=None,
        )
        total_rows = count_df['cnt'].iloc[0]
    except Exception as e:
        print(f"❌ Could not get count: {e}")
        print("📥 Reading all data at once instead...")
        return pandas_gbq.read_gbq(
            processed_query,
            project_id=project_id,
            use_bqstorage_api=True,
            progress_bar_type='tqdm',
            **kwargs
        )
    
    print(f"✅ Total rows to read: {total_rows:,}")
    
    # Calculate number of chunks
    num_chunks = int(np.ceil(total_rows / chunk_size))
    print(f"📦 Creating {num_chunks} chunks of ~{chunk_size:,} rows each...")
    
    # Create chunk parameters
    chunks = []
    for i in range(num_chunks):
        start = i * chunk_size + 1
        end = min((i + 1) * chunk_size, total_rows)
        chunks.append({
            'start_row': start,
            'end_row': end
        })
    
    # Read chunks in parallel with progress bar
    results = []
    print(f"\n🚀 Starting parallel read with {max_workers} workers...\n")
    
    start_time = time.time()
    
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        # Submit all tasks
        futures = {
            executor.submit(
                read_bq_chunk, 
                processed_query, 
                project_id, 
                chunk, 
                idx + 1, 
                num_chunks, 
                **kwargs
            ): idx 
            for idx, chunk in enumerate(chunks)
        }
        
        # Process results with progress bar
        with tqdm(total=num_chunks, desc="Reading chunks", unit="chunk") as pbar:
            for future in as_completed(futures):
                chunk_result = future.result()
                
                if chunk_result['success']:
                    if not chunk_result['result'].empty:
                        results.append(chunk_result['result'])
                        pbar.set_postfix({
                            'rows': f"{chunk_result['rows']:,}",
                            'total': f"{sum(len(r) for r in results):,}"
                        })
                    else:
                        pbar.write(f"⚠️  Chunk {chunk_result['chunk_num']}: Empty result")
                else:
                    pbar.write(f"❌ {chunk_result['message']}")
                
                pbar.update(1)
    
    elapsed_time = time.time() - start_time
    
    # Combine all results
    if results:
        print(f"\n📊 Combining {len(results)} chunks...")
        final_df = pd.concat(results, ignore_index=True)
        
        print(f"\n{'='*60}")
        print(f"✅ COMPLETE: {len(final_df):,} rows read")
        print(f"⏱️  Time elapsed: {elapsed_time:.2f} seconds")
        print(f"📈 Reading speed: {len(final_df)/elapsed_time:.0f} rows/second")
        print(f"📦 Expected rows: {total_rows:,}")
        print(f"📊 Difference: {total_rows - len(final_df):,}")
        print(f"{'='*60}")
        
        return final_df
    else:
        print("⚠️  No data returned")
        return pd.DataFrame()

# Usage with progress tracking
print("="*60)
print("PARALLEL BIGQUERY READ WITH PROGRESS TRACKING")
print("="*60)

df = read_bigquery_parallel(
    sql,
    user_constants,
    project_id="anbc-dev-hcm-cm-de",
    max_workers=6,
    chunk_size=100000,
    configuration={
        'query': {
            'useQueryCache': False,
            'useLegacySql': False,
        }
    },
    dialect='standard',
    auth_local_webserver=False
)

# Check for duplicates
print("\n" + "="*60)
print("DUPLICATE CHECK")
print("="*60)
print(f"📊 Total rows: {len(df):,}")
print(f"✓  Unique rows: {len(df.drop_duplicates()):,}")
print(f"⚠️  Duplicates: {len(df) - len(df.drop_duplicates()):,}")
print(f"📈 Duplicate percentage: {100 * (len(df) - len(df.drop_duplicates())) / len(df):.2f}%")

if df.duplicated().sum() > 0:
    print(f"\n⚠️  Found {df.duplicated().sum()} duplicate rows")
    if df.duplicated().sum() < 100:
        print("\nSample duplicate rows:")
        dups = df[df.duplicated(keep=False)].sort_values(df.columns.tolist())
        print(dups.head(10))

print(f"\n✅ Final shape: {df.shape}")

PARALLEL BIGQUERY READ WITH PROGRESS TRACKING
🔍 Counting total rows...
✅ Total rows to read: 1,374,123
📦 Creating 14 chunks of ~100,000 rows each...

🚀 Starting parallel read with 6 workers...



Reading chunks:   0%|          | 0/14 [00:00<?, ?chunk/s]

❌ Chunk 13/14 failed: The credentials have been revoked or expired, please re-run the application to re-authorize: Reauthentication is needed. Please run `gcloud auth application-default login` to reauthenticate.
❌ Chunk 14/14 failed: The credentials have been revoked or expired, please re-run the application to re-authorize: Reauthentication is needed. Please run `gcloud auth application-default login` to reauthenticate.


In [17]:
constants = {
    "EMAIL": "sahil_gadge_aetna_com", # e.g., john.doe@cvshealth.com
    "COSTCENTER": "13070", # Insert your costcenter here
    "TENANT": "hcm-cm-de", # e.g., mleng-platform
    "USE_COMPUTE_PROJECT": True, # e.g., True or False
    "OWNER": "sahil_gadge_aetna_com",
    "COMPUTE_PROJECT": "anbc-dev-hcm-cm-de",
    "PROJECT": "anbc-dev-hcm-cm-de",
    'LABELS': {'owner': 'sahil_gadge_aetna_com',
            'costcenter': '13070',
            'tenant': 'hcm-cm-de',
            'self_serve': 'true',
            'lob': 'hcb',
            'pipeline_type': 'training_prediction'},
    # 'hcb' or 'pss' or 'ent'
    "LOB": "hcb",
    "DOCKER_URI": "us-docker.pkg.dev/vertex-ai/training/xgboost-cpu.2-1:latest",
    "SERVICE_ACCOUNT": "gchcb-hcm-cm-de-ontpd@anbc-dev-hcm-cm-de.iam.gserviceaccount.com",
    "SERVICE_ACCOUNT_COMPUTE_PROJECT": None,  # Add this key
    "CMEK_KEY": "projects/cvs-key-vault-nonprod/locations/us-east4/keyRings/gkr-nonprod-us-east4/cryptoKeys/gk-anbc-dev-hcm-cm-de-us-east4",
    "CMEK_KEY_COMPUTE_PROJECT": None,  # Add this key
    "COMPUTE_PROJECT": None,  # Add this key
    "SHARED_PROJECT": "anbc-hcb-dev",
    "LOCATION": "us-east4",
    "pipeline_root": "gs://hcm-cm-de-code-hcb-dev/vertex-test/",
    # save_location -> may be helpful for tracking artifacts / model registration etc
    # 'MANUAL_SAVE_LOCATION': 'gs://',
    # name of this repo
    # If defined, overwrites system-derived REPO_NAME value
    "MODEL_DESCRIPTION": "maternity_09_25", # Description for model when registered with ML Platform
    "PIPELINE_TYPE": "training_prediction", # Supported values: training, prediction, evaluation, training_prediction, other
    
    # SQL Variables for BigQuery queries
    "GCP_PROJECT": "anbc-hcb-dev",
    "GCP_DB": "cm_medicaid_hcb_dev", 
    "PREFIX": "a974930_sahil_test",
    "DEFAULT_EXP": "INTERVAL 2 DAY",
    "ST": "{GCP_PROJECT}.{GCP_DB}.{PREFIX}_st",
    "SDOH_YEAR":'2023',
    'JOB_CONFIG': {'cost_center': '13070',
                'email': 'sahil_gadge_aetna_com',
                'project': 'anbc-dev-hcm-cm-de',
                'bq_job_labels': {'self_serve': 'true',
                                  'model_name': 'didactic-broccoli-hcm-cm-de-model',
                                  'pipeline_type': 'training_prediction',
                                  'vertex-ai-pipelines-run-billing-id': 'none'},
                'bq_table_labels': {'self_serve': 'true',
                                    'model_name': 'didactic-broccoli-hcm-cm-de-model',
                                    'pipeline_type': 'training_prediction',
                                    'vertex-ai-pipelines-run-billing-id': 'none'}}
}

#### Define pipeline dag

In [ ]:
@dsl.pipeline(
    name="vertex-hyperparameter-tuning-pipeline",
    description="Pipeline for Vertex AI hyperparameter tuning with containerized component"
)
def hptune_pipeline(
    # Core hyperparameter tuning parameters
    project: str,
    service_account: str,
    cmek_key: str,
    location:str,
    pipeline_root:str,
    model_dir:str,
    # Hyper tune configuration
    hptune_image_uri: str ,
    hptune_machine_type: dict ,
    hptune_package_uris:list,
    hptune_python_module:str,
    hptune_training_args: list,
    parameter_spec_dict: dict,
    eval_metrics: dict,
    max_trials: int,
    parallel_trials: int,
    max_failed_trials: int,
    # Retrain configuration
    training_image_uri: str,
    training_machine_type: dict,
    training_package_uris:list,
    training_python_module:str,
    training_args: list,
    # Register
    model_display_name: str="new_model",
    serving_image_uri: str = "us-docker.pkg.dev/vertex-ai/prediction/xgboost-cpu.1-6:latest",
    model_description: str = None,
    # Pipeline configuration
    upload_to_existing_model: bool=False,
    existing_model_resource_name: str = None,
):


    # Call the hyperparameter tuning component
    hptune_job = vertex_hyperparameter_tuning_component(
        project=project,
        service_account=service_account,
        cmek_key=cmek_key,
        location=location,
        pipeline_root=pipeline_root,
        image_uri=hptune_image_uri,
        package_uris=hptune_package_uris,
        python_module=hptune_python_module,
        training_args=hptune_training_args,
        parameter_spec_dict=parameter_spec_dict,
        eval_metrics=eval_metrics,
        machine_type=hptune_machine_type,
        parallel_trials=parallel_trials,
        max_trials=max_trials,
        max_failed_trials=max_failed_trials,
    )


#### Compile pipeline

In [15]:
compiler.Compiler().compile(
    pipeline_func=hptune_pipeline,
    package_path="hptune_pipeline_runtime_param_package.yaml"
)
print("Pipeline compiled to hptune_pipeline_runtime_param_package.yaml")

Pipeline compiled to hptune_pipeline_runtime_param_package.yaml


#### Input for the parameter in a dictionary

In [16]:
pipeline_root=constants["pipeline_root"]
pipeline_name = "vertex-hptune-pipeline-runtime-param-package" 

parameter_values={
    "project": constants["PROJECT"],
     "service_account": constants["SERVICE_ACCOUNT"],
     "cmek_key": constants["CMEK_KEY"],
     "location": constants["LOCATION"],
    "pipeline_root":constants["pipeline_root"],
    "model_dir":f"{pipeline_root}/model",
    "training_package_uris":["gs://hcm-cm-de-data-hcb-dev/mlops-pipeline-test/trainer-0.1.tar.gz"],
    "training_python_module":"trainer.task",
    "training_args":[ "--project", constants["PROJECT"],
                      "--training_table", "anbc-hcb-dev.cm_medicaid_hcb_dev.a974930_sahil_test_selected_features_train",  # Replace with your BigQuery table name
                      "--target_column", "y",  
                      "--model_dir", f"{pipeline_root}/model"  # GCS path to save the model
                 ],
    "hptune_package_uris":["gs://hcm-cm-de-data-hcb-dev/mlops-pipeline-test/trainer-0.1.tar.gz"],
    "hptune_python_module":"trainer.task",
    "hptune_training_args":[ "--project", constants["PROJECT"],
                      "--training_table", "anbc-hcb-dev.cm_medicaid_hcb_dev.a974930_sahil_test_selected_features_train",  # Replace with your BigQuery table name
                      "--target_column", "diabetes",  
                      "--model_dir", f"{pipeline_root}/model"  # GCS path to save the model
                 ],
    "parameter_spec_dict": {
        "eta": {"type": "double", "min": 0.01, "max": 0.3, "scale": "log"},
        "max_depth": {"type": "integer", "min": 3, "max": 10, "scale": "linear"}
    },
    "eval_metrics": {"roc_auc":"maximize"},  # No serialization
    "max_trials": 10,
    "parallel_trials": 2,
    "max_failed_trials": 2,
    "training_image_uri": "us-docker.pkg.dev/vertex-ai/training/xgboost-cpu.1-6",
    "training_machine_type": {"machine_type": "n1-standard-16",
                        "accelerator_type": None,  # string if not None
                        "accelerator_count": None,  # int if not None
                    },
    "hptune_image_uri": "us-docker.pkg.dev/vertex-ai/training/xgboost-cpu.2-1:latest",
    "hptune_machine_type": {"machine_type": "n1-standard-16",
                        "accelerator_type": None,  # string if not None
                        "accelerator_count": None,  # int if not None
                    }
}
aiplatform.init(project=constants["PROJECT"], location=constants["LOCATION"],service_account=constants["SERVICE_ACCOUNT"])

pipeline_job = aiplatform.PipelineJob(
    display_name=pipeline_name,
    template_path="hptune_pipeline_runtime_param_package.yaml",
    pipeline_root=pipeline_root,
    parameter_values=parameter_values,
    enable_caching=False,
    encryption_spec_key_name=constants["CMEK_KEY"],
)

pipeline_job.run(sync=True)

ValueError: The pipeline parameter model_dir is not found in the pipeline job input definitions.

#### create run with the local compiled yaml template

In [ ]:
aiplatform.init(project=constants["project"], location=constants["location"],service_account=constants["service_account"])

pipeline_job = aiplatform.PipelineJob(
    display_name=pipeline_name,
    template_path="hptune_pipeline_runtime_param_package.yaml",
    pipeline_root=pipeline_root,
    parameter_values=parameter_values,
    enable_caching=False,
    encryption_spec_key_name=constants["cmek_key"],
)

pipeline_job.run(sync=True)

#### Or 1) Upload template to artifect refgistry

In [ ]:
from kfp.registry import RegistryClient
client = RegistryClient(host=f"https://us-east4-kfp.pkg.dev/anbc-dev-hcm-cm-de/hcm-cm-de-kfp-repo")

templateName, versionName = client.upload_pipeline(
  file_name="hptune_pipeline_runtime_param_package.yaml",
  tags=["v3"],
  extra_headers={"description":"This is an hyper tune training pipeline template."})

#### 2) create run on uploaded template with version and tag

In [ ]:
aiplatform.init(project=constants["project"], location=constants["location"],service_account=constants["service_account"])

job = aiplatform.PipelineJob(
    display_name="hypertune",
    template_path=f"https://us-east4-kfp.pkg.dev/anbc-dev-hcm-cm-de/hcm-cm-de-kfp-repo/{pipeline_name}/"+"v3",
    parameter_values=parameter_values
)
job.run(sync=True)

### Batch inference pipeline

In [ ]:
@dsl.pipeline(
    name="batch-inference-pipeline",
    description="Pipeline for Vertex AI batch inference with a model in model registry and input and output in BigQuery"
)
def batch_inference_pipeline(
    project: str,
    location: str,
    service_account: str,
    cmek_key: str,
    cost_center: str,
    owner: str,
    #Model details
    model_resource_name: str,
    # BigQuery specific
    key_field: str,  # Unique key field in input table
    input_table: str,  # project.dataset.table
    output_table: str, # project.dataset.table (final output)
    compute_dataset: str, #hcm_cm_de_dec_beam_{ENV}_hcm_cm_de"
    expiration_days: int = 30, # BQ table expiration in days
    # Instance configuration - field filtering
    excluded_fields: list = None,  # Fields to exclude from predictions
    included_fields: list = None,  # Fields to include (if specified, only these will be sent)
    # Machine configuration
    machine_type: dict = {"machine_type": "n2-standard-64"},
    # Job configuration
    starting_replica_count: int = 1,
    max_replica_count: int = 1,
):


    # Call the hyperparameter tuning component
    copy_bq_table = vertex_batch_predict_bigquery_component(
        project=project,
        service_account=service_account,
        cmek_key=cmek_key,
        location=location,
        cost_center= cost_center,
        owner=owner,
        model_resource_name=model_resource_name,
        key_field=key_field,  # Unique key field in input table
        input_table=input_table,  # project.dataset.table
        output_table=output_table, # project.dataset.table (final output)
        compute_dataset=compute_dataset, #hcm_cm_de_dec_beam_{ENV}_hcm_cm_de"
        expiration_days=expiration_days, # BQ table expiration in days
        excluded_fields=excluded_fields,  # Fields to exclude from predictions
        included_fields=included_fields,  # Fields to include (if specified, only these will be sent)
        machine_type=machine_type,
        starting_replica_count=starting_replica_count,
        max_replica_count=max_replica_count,
    )

In [ ]:
compiler.Compiler().compile(
    pipeline_func=batch_inference_pipeline,
    package_path="batch_inference_pipeline.yaml"
)
print("Pipeline compiled to batch_inference_pipeline.yaml")

In [ ]:
parameter_values={
    "project": constants["project"],
     "service_account": constants["service_account"],
     "cmek_key": constants["cmek_key"],
     "location": constants["LOCATION"],
     "cost_center":constants["costcenter"],
     "owner":constants["owner"],
    "model_resource_name": "projects/anbc-dev-hcm-cm-de/locations/us-east4/models/2195280517971050496@3",
    # BigQuery specific
    "key_field": "individual_id",  # Unique key field in input table
    "input_table": "anbc-hcb-dev.clin_analytics_hcb_dev.mlops_pipeline_test_falls_training_sample",  # project.dataset.table
    "output_table": "anbc-hcb-dev.clin_analytics_hcb_dev.mlops_test_falls_prediction_v3", # project.dataset.table (final output)
    "compute_dataset": "hcm_cm_de_beam_dev_hcm_cm_de", #hcm_cm_de_dec_beam_{ENV}_hcm_cm_de"
    "expiration_days": 30, # BQ table expiration in days
    # Instance configuration - field filtering
    "excluded_fields":["individual_id","y"] ,  # Fields to exclude from predictions
    # Machine configuration
    "machine_type": {"machine_type": "n1-standard-16"},
    # Job configuration
    "starting_replica_count": 1,
    "max_replica_count": 1,
}
aiplatform.init(project=constants["project"], location=constants["LOCATION"],service_account=constants["service_account"])

pipeline_job = aiplatform.PipelineJob(
    display_name="batch_inference_pipeline" ,
    template_path="batch_inference_pipeline.yaml",
    pipeline_root=constants["pipeline_root"],
    parameter_values=parameter_values,
    # enable_caching=False,
    encryption_spec_key_name=constants["cmek_key"],
)
pipeline_job.run(sync=True)

In [1]:
import pyspark

ModuleNotFoundError: No module named 'pyspark'